In [ ]:
#1,2,7,15,16,17
class Player:
    def __init__(self,id,name,hp):
        self._id=id
        self._name=name.strip().title()
        self._hp=hp if hp>0 else 0
        self._inventory=Inventory()
    def __str__(self):
        return f"Player(id={self._id}, name='{self._name}', hp={self._hp})"
    def __del__(self):
        print(f"Player <{self._name}> удалён")
    @classmethod
    def from_string(cls, data: str):
        if not isinstance(data,str):
            raise ValueError
        a=[x.strip() for x in data.split(',')]
        if len(a)!=3:
            raise ValueError
        try:
            id=int(a[0])
            name = a[1]
            hp= int(a[2])
        except:
            raise ValueError
        return cls(id,name,hp)
    @property
    def id(self):
        return self._id
    @property
    def hp(self):
        return self._hp
    @property
    def inventory(self):
        return self._inventory

    def handle_event(self,event:Event):
        if event.type=='ATTACK':
            self._hp-=event.data.get('damage',0)
        elif event.type=='HEAL':
            self._hp+=event.data.get('heal',0)
        elif event.type=='LOOT':
            self._inventory.add_item(event.data.get('item'))



class Warrior(Player):
    def handle_event(self,event:Event):
        if event.type=='ATTACK':
            self._hp-=event.data.get('damage',0)*0.9
        else:
            super().handle_event(event)

class Mage(Player):
    def handle_event(self,event:Event):
        if event.type=='LOOT':
            i=event.data.get('item')
            i.power=int(i.ppwer*1.1)
        super().handle_event(event)

In [2]:
#3
class Item:
    def __init__(self,id,name,power):
        self.id=id
        self.name=name.strip().title()
        self.power=power

    def __str__(self):
        return f"Item(id={self.id},name='{self.name}',power={self.power})"
    def __eq__(self,other):
        if not isinstance(other,Item):
            return False
        return self.id==other.id
    def __repr__(self):
        return self.__str__()
    def __hash__(self):
        return hash(self.id)



In [1]:
#4,5,18
class Inventory:
    def __init__(self):
        self._items:Dict[int,Item]={}
    def add_item(self,item: Item):
        self._items[item.id]= item
    def remove_item(self ,item_id: int):
        self._items.pop(item_id,None)
    def get_items(self) -> list[Item]:
        return list(self._items.values())
    def unique_items(self)-> set[Item]:
        return set(self._items.values())
    def to_dict(self) -> dict[int, Item]:
        return self._items.copy()
    def get_strong_items(self,min_power:int)->list[Item]:
        r=lambda x: x.power>=min_power
        return [i for i in self._items.values() if r(i)]
    def __iter__(self):
        return iter(self._items.values())


In [3]:
#6
from datetime import datetime
class Event:
    def __init__(self,type:str,data:dict):
        self.type=type
        self.data=data
        self.timestamp= datetime.now()
    def __str__(self):
        return f"Event(type='{self.type}',data={self.data},timestamp='{self.timestamp}'"


In [4]:
#8,9
class Logger:
    @staticmethod
    def log(event: Event, player: Player, filename: str):
        with open(filename,'a') as f:
            f.write(f"{event.timestamp}: {player.id};{event.type};{event.data}\n")
    @staticmethod
    def read_logs(filename: str) -> list[Event]:
        a=[]
        with open(filename,'r') as f:
            for x in f:
                p=x.strip().split(';')
                e=Event(p[2],eval(p[3]))
                e.timestamp=p[0]
                a.append(e)
        return a


In [6]:
#10
class EventIterator:
    def __init__(self,events):
        self.events=events
        self.index=0
    def __iter__(self):
        return self
    def __next__(self):
        if self.index>=len(self.events):
            raise StopIteration
        e=self.events[self.index]
        self.index+=1
        return e
